In [ ]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

REQUIRE_CUDA = True  # set False for CPU-only notebooks

print(f"CPU threads set to: {CPU_THREADS}")

try:
    import torch
except Exception as e:
    torch = None
    if REQUIRE_CUDA:
        raise RuntimeError("CUDA required but torch is not available.") from e

if torch is not None:
    torch.set_num_threads(CPU_THREADS)
    torch.set_num_interop_threads(min(4, CPU_THREADS))
    if REQUIRE_CUDA and not torch.cuda.is_available():
        raise RuntimeError("CUDA required but not available.")
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available; running on CPU.")


# Kvasir-VQA x1 — BLIP-2 VQA fine-tuning

Fine-tune a stronger VQA transformer (BLIP-2 / InstructBLIP) on the x1 splits. The notebook mirrors the BLIP baseline but upgrades the backbone and keeps outputs under `2_modeling/06_blip2_finetune/out/`.

In [1]:
import os
# Avoid TF/Keras import issues in transformers (this notebook uses PyTorch)
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset

from transformers import Blip2Processor, Blip2ForConditionalGeneration, BlipProcessor, BlipForQuestionAnswering
from transformers import TrainingArguments, Trainer
import evaluate


/workspace/vqa-rag/lib/python3.12/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
# Paths & config

def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "06_blip2_finetune" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Swap to another backbone if you prefer (e.g., 'Salesforce/instructblip-vicuna-7b' for instruction-tuned VQA)
MODEL_FAMILY = "blip"  # "blip" (smaller) or "blip2"
if MODEL_FAMILY == "blip":
    MODEL_NAME = "Salesforce/blip-vqa-base"
    PROMPT_TEMPLATE = "{question}"
    USE_8BIT = False
    DEVICE_MAP = None
    TORCH_DTYPE = torch.float32
    GRADIENT_CHECKPOINTING = False
else:
    MODEL_NAME = "Salesforce/blip2-flan-t5-xl"
    PROMPT_TEMPLATE = "Question: {question}\nAnswer:"
    USE_8BIT = True  # set False if bitsandbytes not available
    DEVICE_MAP = "auto"  # use HF accelerate device placement; set None to disable
    USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    TORCH_DTYPE = torch.bfloat16 if USE_BF16 else (torch.float16 if torch.cuda.is_available() else torch.float32)
    GRADIENT_CHECKPOINTING = True
# AMP (fp16/bf16) flags are set later based on actual model dtype
FORCE_CPU = False  # set True to force CPU
USE_GPU = torch.cuda.is_available() and not FORCE_CPU
if USE_GPU:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

DEVICE = torch.device("cuda" if USE_GPU else "cpu")
SEED = 42
MAX_ANSWER_LEN = 16
QUESTION_MAX_LEN = 64
MAX_GEN_TOKENS = 12

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS = 1
LR = 1e-5

MAX_TRAIN_SAMPLES = None  # set int for smoke tests
MAX_VAL_SAMPLES = None
MAX_TEST_SAMPLES = None


torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Out dir:", OUT_DIR)
print("Device:", DEVICE)


Data root: /workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Out dir: /workspace/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/06_blip2_finetune/out
Device: cpu


In [3]:
# Load metadata
meta = pd.read_csv(META_CSV)

images_base = DATA_ROOT / "0_dataset_prep"
meta["image_path"] = meta["image_path"].apply(
    lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p
)

if "split" not in meta.columns:
    raise RuntimeError("Missing 'split' column. Run dataset prep split step first.")

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "validation"].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

# Drop rows missing answers
train_df = train_df.dropna(subset=["answer"]).reset_index(drop=True)
val_df = val_df.dropna(subset=["answer"]).reset_index(drop=True)
test_df = test_df.dropna(subset=["answer"]).reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


{'train': 46923, 'val': 5928, 'test': 5947}


In [4]:
# Optional subsampling
if MAX_TRAIN_SAMPLES is not None:
    train_df = train_df.sample(min(MAX_TRAIN_SAMPLES, len(train_df)), random_state=SEED).reset_index(drop=True)
if MAX_VAL_SAMPLES is not None:
    val_df = val_df.sample(min(MAX_VAL_SAMPLES, len(val_df)), random_state=SEED).reset_index(drop=True)
if MAX_TEST_SAMPLES is not None:
    test_df = test_df.sample(min(MAX_TEST_SAMPLES, len(test_df)), random_state=SEED).reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})

{'train': 46923, 'val': 5928, 'test': 5947}


In [5]:
# Load processor + model
if MODEL_FAMILY == "blip2":
    processor = Blip2Processor.from_pretrained(MODEL_NAME)

    bnb_config = None
    if USE_8BIT:
        try:
            from transformers import BitsAndBytesConfig
            bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        except Exception as e:
            print("8-bit quantization unavailable, falling back to full precision:", e)

    if bnb_config is not None:
        try:
            model = Blip2ForConditionalGeneration.from_pretrained(
                MODEL_NAME,
                quantization_config=bnb_config,
                device_map=DEVICE_MAP,
            )
        except Exception as e:
            print("8-bit load failed, falling back to full precision:", e)
            bnb_config = None

    if bnb_config is None:
        model = Blip2ForConditionalGeneration.from_pretrained(
            MODEL_NAME,
            torch_dtype=TORCH_DTYPE,
            device_map=DEVICE_MAP if DEVICE_MAP is not None else None,
        )
        if DEVICE_MAP is None:
            model.to(DEVICE)
else:
    processor = BlipProcessor.from_pretrained(MODEL_NAME)
    model = BlipForQuestionAnswering.from_pretrained(MODEL_NAME)
    model.to(DEVICE)

if GRADIENT_CHECKPOINTING and hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "right"
if hasattr(model.config, "text_config"):
    model.config.text_config.pad_token_id = processor.tokenizer.pad_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

In [6]:
class VQADataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        prompt = PROMPT_TEMPLATE.format(question=str(row["question"]))
        inputs = processor(images=image, text=prompt, return_tensors="pt", padding="max_length", truncation=True, max_length=QUESTION_MAX_LEN)
        labels = processor.tokenizer(
            str(row["answer"]) if pd.notna(row["answer"]) else "",
            max_length=MAX_ANSWER_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        ).input_ids.squeeze(0)
        labels[labels == processor.tokenizer.pad_token_id] = -100
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        item["labels"] = labels
        return item

def collate_fn(batch):
    keys = batch[0].keys()
    return {k: torch.stack([b[k] for b in batch]) for k in keys}

train_ds = VQADataset(train_df)
val_ds = VQADataset(val_df)
test_ds = VQADataset(test_df)

print("Sample prompt:", PROMPT_TEMPLATE.format(question=train_df.iloc[0]["question"]))

Sample prompt: Are there any abnormalities in the image? Check all that are present.


In [7]:
class VQATrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Drop HF's helper arg that T5 models don't accept
        inputs = dict(inputs)
        inputs.pop("num_items_in_batch", None)
        labels = inputs.pop("labels", None)
        outputs = model(**inputs, labels=labels)
        loss = outputs.loss if hasattr(outputs, "loss") else outputs["loss"]
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=str(OUT_DIR / "checkpoints"),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    use_cpu=not USE_GPU,
    fp16=False,
    bf16=False,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    remove_unused_columns=False,
    report_to=[],
)

trainer = VQATrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
)

print(trainer)


In [8]:
# Train
trainer.train()

# Save final model
final_dir = OUT_DIR / "final_model"
trainer.save_model(final_dir)
processor.save_pretrained(final_dir)
print("Saved model to", final_dir)

IndexError: index out of range in self

In [ ]:
# Evaluate with BLEU/ROUGE on test split
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

model.eval()

def generate_answer(row):
    img = Image.open(row["image_path"]).convert("RGB")
    prompt = PROMPT_TEMPLATE.format(question=str(row["question"]))
    inputs = processor(images=img, text=prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_GEN_TOKENS)
    return processor.tokenizer.decode(out[0], skip_special_tokens=True).strip()

preds = []
refs = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="BLIP-2 eval"):
    preds.append(generate_answer(row))
    refs.append(str(row["answer"]))

bleu_refs = [[r] for r in refs]
bleu_score = bleu.compute(predictions=preds, references=bleu_refs)["bleu"]
rouge_l = rouge.compute(predictions=preds, references=refs)["rougeL"]

results = {"bleu": bleu_score, "rougeL": rouge_l}
with open(OUT_DIR / "metrics_test.json", "w") as f:
    json.dump(results, f, indent=2)

print(results)